In [0]:
from pyspark.sql import functions as F

In [0]:
CATALOG = "clinical_trial_supply_chain"

SILVER = f"{CATALOG}.silver"
GOLD = f"{CATALOG}.gold"

INVENTORY = f"{SILVER}.fact_inventory"
CONSUMPTION = f"{SILVER}.fact_daily_consumption"
DRUG = f"{SILVER}.dim_drug"
SITE = f"{SILVER}.dim_site_scd2"

TARGET = f"{GOLD}.gold_stockout_risk"

In [0]:
avg_consumption_df = (
    spark.table(CONSUMPTION)
    .groupBy(
        "site_id",
        "drug_type"
    )
    .agg(
        F.avg(
            "total_quantity_dispensed"
        ).alias(
            "avg_daily_consumption"
        )
    )
)

In [0]:
inventory_df = (
    spark.table(INVENTORY)
    .groupBy(
        "site_id",
        "drug_type"
    )
    .agg(
        F.max("snapshot_date")
        .alias("latest_snapshot")
    )
)

In [0]:
latest_inventory_df = (
    spark.table(INVENTORY).alias("i")
    .join(
        inventory_df.alias("m"),
        (
            (F.col("i.site_id") == F.col("m.site_id"))
            &
            (F.col("i.drug_type") == F.col("m.drug_type"))
            &
            (
                F.col("i.snapshot_date")
                ==
                F.col("m.latest_snapshot")
            )
        ),
        "inner"
    )
    .select(
        F.col("i.site_id"),
        F.col("i.study_id"),
        F.col("i.drug_type"),
        F.col("i.snapshot_date"),
        F.col("i.qty_on_hand"),
        F.col("i.qty_reserved"),
        F.col("i.qty_expired"),
        F.col("i.available_stock"),
        F.col("i.inventory_adjustment")
    )
)

In [0]:
stockout_df = (
    latest_inventory_df
    .join(
        avg_consumption_df,
        ["site_id", "drug_type"],
        "left"
    )
    .join(
        spark.table(DRUG)
             .select(
                 "drug_type",
                 "reorder_threshold"
             ),
        "drug_type",
        "left"
    )
    .join(
        spark.table(SITE)
             .filter("is_current = true")
             .select(
                 "site_id",
                 "site_name"
             ),
        "site_id",
        "left"
    )
)

In [0]:
stockout_df.printSchema()

In [0]:
stockout_df = (
    stockout_df
    .withColumn(
        "avg_daily_consumption",
        F.coalesce(
            F.col("avg_daily_consumption"),
            F.lit(0.0)
        )
    )
    .withColumn(
        "days_of_supply_remaining",
        F.when(
            F.col("avg_daily_consumption") > 0,
            F.col("available_stock")
            /
            F.col("avg_daily_consumption")
        )
    )
    .withColumn(
        "stockout_risk_flag",
        F.when(
            F.col("days_of_supply_remaining")
            <=
            F.col("reorder_threshold"),
            "HIGH_RISK"
        )
        .otherwise("LOW_RISK")
    )
)

In [0]:
(
    stockout_df.select(
        "site_id",
        "site_name",
        "drug_type",
        "available_stock",
        "avg_daily_consumption",
        "days_of_supply_remaining",
        "reorder_threshold",
        "stockout_risk_flag"
    )
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TARGET)
)